# ⚖️ DLKR-LJP: Dynamic Legal Knowledge Retrieval for Indian Legal Judgment Prediction
### Complete Self-Contained Google Colab Runner (Free T4 GPU Optimized)

This notebook runs the **DLKR-LJP v4.0** pipeline on a free Google Colab T4 GPU instance.

**Core Architecture:**
1. **Pretrained Legal Reasoner:** `Qwen2.5-3B-Instruct` (4-bit quantized, ~2.2 GB VRAM) for zero-shot structured legal fact extraction.
2. **Pretrained Legal Backbone:** `law-ai/InLegalBERT` (110M params, fp16) by IIT Kharagpur for deep Indian statutory representation.
3. **Dynamic Retrieval Module (DRM):** Lexical + Dense Semantic Embeddings (`all-MiniLM-L6-v2` / `BGE`).
4. **Relevance & Confidence Gating (RCG):** Dual-check matrix filtering non-relevant statutory provisions.
5. **Judicial Consistency Verifier (JCV):** Audits real substantive conflicts (Section 302 vs 304A, Section 392 vs 379, Solitary Accused s.34 defects, and BNS 2023 / IPC 1860 penalty ceilings).

## 🛠️ Step 1: Check GPU Acceleration
Ensure your Colab runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [1]:
!nvidia-smi

Tue Oct 13 15:19:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 📦 Step 2: Install Required Libraries

In [2]:
!pip install -q gradio transformers torch sentence-transformers accelerate bitsandbytes chromadb datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

## 🔑 Step 2.5: Configure API Key (Optional for Zero-Shot Semantic LLM)
Set `GEMINI_API_KEY` (Free from Google AI Studio) or `GROQ_API_KEY` (Free from Groq Cloud) to enable cloud zero-shot semantic comprehension for arbitrary natural language phrasing without regex fragility.
In Colab, you can also store this under the 🔑 Secrets tab.

In [3]:
import os
from pathlib import Path

# 1. Check Colab Secrets (Left sidebar 🔑)
try:
    from google.colab import userdata
    val = userdata.get('GEMINI_API_KEY')
    if val:
        os.environ['GEMINI_API_KEY'] = val
except Exception:
    pass

# 2. Check uploaded .env file (/content/.env or .env)
for p in [Path('/content/.env'), Path('.env')]:
    if not os.environ.get('GEMINI_API_KEY') and p.exists():
        for line in p.read_text(encoding='utf-8').splitlines():
            line_str = line.strip()
            if line_str.startswith('GEMINI_API_KEY='):
                os.environ['GEMINI_API_KEY'] = line_str.split('=', 1)[1].strip().strip('"').strip("'")

# 3. Set your active key directly
if not os.environ.get('GEMINI_API_KEY'):
    os.environ['GEMINI_API_KEY'] = ''

is_active = bool(os.environ.get('GEMINI_API_KEY') or os.environ.get('GROQ_API_KEY'))
print('Zero-Shot LLM API Active:', is_active)
if is_active:
    print('✅ Successfully connected to Zero-Shot Semantic LLM Reasoning Engine.')
else:
    print('⚠️ API Key not detected. Running on deterministic statutory prober.')


Zero-Shot LLM API Active: True
✅ Successfully connected to Zero-Shot Semantic LLM Reasoning Engine.


## 💾 Step 3: Mount Google Drive (Optional for Weight Persistence)
Mounting Google Drive ensures that trained `legal_bert_heads.pt` weights and checkpoints persist across Colab session disconnects.

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=False)
os.makedirs('/content/drive/MyDrive/DLKR_LJP', exist_ok=True)
print("✅ Google Drive successfully configured for permanent checkpoint storage.")

Mounted at /content/drive
✅ Google Drive successfully configured for permanent checkpoint storage.


## 🚀 Step 4: Run the Complete DLKR-LJP Dashboard
Make sure `standalone_dlkr_ljp (4).py` is uploaded to `/content/` (or run in current directory).
This launches the interactive Gradio web app with a **public shareable URL** (`https://xxxx.gradio.live`).

In [ ]:
!python "standalone_dlkr_ljp (4).py"

/content/standalone_dlkr_ljp (4).py:4757: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="DLKR-LJP: Legal Judgment Prediction", theme=theme) as demo:
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://c5922795c5c641d92e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
modules.json: 100% 349/349 [00:00<00:00, 1.36MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 638kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 31.0MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 65.8kB/s]
config.json: 100% 612/612 [00:00<00:00, 4.04MB/s]

model.safetensors: downloading bytes:  87% 78.9M/90.9M [00

## 🧪 Step 5: Direct Python API Adjudication (Optional)
You can also invoke the pipeline directly in notebook cells for automated evaluation or batch test cases.

In [ ]:
import importlib.util
import sys

# Import pipeline
spec = importlib.util.spec_from_file_location('dlkr', 'standalone_dlkr_ljp (4).py')
mod = importlib.util.module_from_spec(spec)
sys.modules['dlkr'] = mod
spec.loader.exec_module(mod)

pipeline = mod.get_pipeline()

# Example narrative
case_narrative = "The accused armed with a large curved knife accosted the commuter on an isolated street at night, brandished the blade at his throat, and forcibly extorted his wallet and cash under threat of instant death."

result = pipeline.run(case_narrative)
preds = result['predictions']

print("=" * 60)
print("⚖️ DLKR-LJP ADJUDICATION REPORT")
print("=" * 60)
print(f"📌 Primary Law Article : {preds['head1_articles'][0]['display_name']}")
print(f"   Confidence & Status  : {preds['head1_articles'][0]['probability']:.1%} ({preds['head1_articles'][0].get('support_type', 'N/A')})")
print(f"📌 Criminal Charge      : {preds['head2_charges'][0]['charge_name']}")
print(f"📌 Sentencing Severity  : {preds['primary_penalty']['tier']}")
print(f"📌 JCV Statutory Audit  : {preds['judicial_consistency_audit']['audit_status']} (JCI: {preds['judicial_consistency_audit']['judicial_consistency_index']}%)")
print("=" * 60)